In [0]:
dbutils.widgets.text("source_name", "")

In [0]:
source_name = dbutils.widgets.get("source_name")
print(source_name)

orders


In [0]:
## Read Config Table Based on Widget Value
config = (
    spark.table("dev.demo.ingestion_config")
    .filter(f"source_name = '{source_name}'")
    .first()
)

In [0]:
if not source_name:
    raise ValueError(
        "source_name parameter is mandatory. "
        "Pass it from Workflow or set it manually for testing."
    )

if config is None:
    raise ValueError(
        f"No configuration found for source_name='{source_name}' "
        f"in dev.demo.ingestion_config."
    )

In [0]:
print("Config loaded for:", source_name)
print(config)

Config loaded for: orders
Row(source_name='orders', source_path='/Volumes/dev/demo/datasets/source/orders', target_path='/Volumes/dev/demo/datasets/bronze/orders', file_format='json', load_type='append')


In [0]:
source_path = config["source_path"]
target_path = config["target_path"]
file_format = config["file_format"]
load_type = config["load_type"]

In [0]:
print("Source Path:", source_path)
print("Target Path:", target_path)
print("File Format:", file_format)
print("Load Type:", load_type)

Source Path: /Volumes/dev/demo/datasets/source/orders
Target Path: /Volumes/dev/demo/datasets/bronze/orders
File Format: json
Load Type: append


In [0]:
## Generic Read Logic
df = (
    spark.read
    .format(file_format)
    .option("recursiveFileLookup", "true")
    .load(source_path)
)

In [0]:
# display(df)

In [0]:
## Add Audit Columns
from pyspark.sql.functions import col, current_timestamp

df = (
    df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
print("Source:", source_path)
print("Target:", target_path)
print("Rows:", df.count())

Source: /Volumes/dev/demo/datasets/source/orders
Target: /Volumes/dev/demo/datasets/bronze/orders
Rows: 8


In [0]:
## Generic Bronze Write
df.write \
    .format("delta") \
    .mode(load_type) \
    .option("mergeSchema", "true") \
    .save(target_path)

In [0]:
print(f"Successfully loaded {source_name}")
print(f"Written to {target_path}")

Successfully loaded orders
Written to /Volumes/dev/demo/datasets/bronze/orders


In [0]:
df = spark.read.format("delta").load("/Volumes/dev/demo/datasets/bronze/orders")
display(df)

customer,order_date,order_id,product,quantity,ingestion_ts,source_file
"List(101, Anil)",2025-01-10,1,"List(1001, Laptop)",1,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00000-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-315-1-c000.json
"List(102, Rahul)",2025-02-11,2,"List(1002, Mobile)",2,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00001-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-316-1-c000.json
"List(103, Priya)",2025-03-12,3,"List(1003, Headphones)",3,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00002-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-311-1-c000.json
"List(101, Anil)",2025-04-13,4,"List(1004, Monitor)",1,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00003-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-317-1-c000.json
"List(101, Anil)",2025-05-14,5,"List(1002, Mobile)",2,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00004-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-312-1-c000.json
"List(102, Rahul)",2025-06-15,6,"List(1001, Laptop)",1,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00005-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-318-1-c000.json
"List(103, Priya)",2025-07-16,7,"List(1004, Monitor)",2,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00006-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-313-1-c000.json
"List(101, Anil)",2025-08-17,8,"List(1003, Headphones)",4,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch1/part-00007-tid-3596166186404612892-24d1d0dc-844c-44cd-8949-93983d950e4e-314-1-c000.json
"List(101, Anil)",2025-09-10,9,"List(1001, Laptop)",1,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch2/part-00000-tid-3597243309734365409-dca154c1-2e5a-4adc-8dc0-e75e5bddc744-171-1-c000.json
"List(102, Rahul)",2025-09-15,10,"List(1004, Monitor)",2,2026-08-26T16:34:30.723Z,dbfs:/Volumes/dev/demo/datasets/source/orders/orders_batch2/part-00001-tid-3597243309734365409-dca154c1-2e5a-4adc-8dc0-e75e5bddc744-172-1-c000.json
